# 📈 Phân tích Cổ phiếu MBB — Bộ câu hỏi nâng cao V3
## Giai đoạn 2021–2026

| # | Câu hỏi | Cách làm | Tại sao |
|---|---------|----------|---------|
| Q1 | Sau khi MBB đạt đỉnh 52 tuần, giá tiếp tục tăng hay đảo chiều? | Tìm ngày đạt đỉnh → xem % thay đổi 5/10/20 phiên sau | Đỉnh 52 tuần là tín hiệu trader theo dõi nhiều nhất |
| Q2 | Tháng nào MBB có nhiều ngày TĂNG nhất (tỷ lệ ngày xanh)? | Đếm ngày tăng/giảm từng tháng → tính % ngày xanh | Biết tháng thuận giúp chọn thời điểm giao dịch tốt hơn |
| Q3 | Bóng nến trên (High–Close) có dự báo ngày hôm sau giảm không? | Tính tỷ lệ bóng trên/biên độ → gán nhãn 1/0 → kiểm chứng | Bóng trên lớn = áp lực bán mạnh, có thể dự báo chiều giá |
| Q4 | Phiên đầu tháng và cuối tháng, phiên nào MBB thường tăng hơn? | Lấy phiên đầu/cuối mỗi tháng → tính % thay đổi → so sánh | Dòng tiền vào/ra cuối tháng tạo pattern có thể khai thác |
| Q5 | Sau 3 phiên TĂNG liên tiếp, MBB có xu hướng đảo chiều không? | Gán nhãn chuỗi 3×1 → xem phiên thứ 4 tăng hay giảm | Kiểm chứng Mean Reversion — quy luật hồi về trung bình |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_style('whitegrid')
print('✅ Import thành công!')

✅ Import thành công!


In [ ]:
df = pd.read_csv('data/dulieu.csv')
df.columns = df.columns.str.strip()
print('Cột gốc :', df.columns.tolist())
print('Date mẫu:', df['Date'].iloc[0])

df.rename(columns={
    'Ngày':'Date','Mở cửa':'Open','Cao nhất':'High',
    'Thấp nhất':'Low','Đóng cửa':'Close',
    'Adj Close':'AdjClose','KL GD':'Volume'
}, inplace=True)

df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=False, errors='coerce')
if df['Date'].isna().sum() > 0:
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True, errors='coerce')

for col in ['Open','High','Low','Close','Volume']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.sort_values('Date').reset_index(drop=True)
df.dropna(subset=['Date','Close','High','Low','Volume'], inplace=True)
df = df.reset_index(drop=True)

df['Year']         = df['Date'].dt.year
df['Month']        = df['Date'].dt.month
df['YearMonth']    = df['Date'].dt.to_period('M')
df['Close_pct']    = df['Close'].pct_change() * 100
df['Range']        = df['High'] - df['Low']
df['Upper_shadow'] = df['High'] - df['Close']
df['Shadow_ratio'] = np.where(df['Range'] > 0, df['Upper_shadow'] / df['Range'], 0)
df['Signal']       = np.where(df['Close_pct'] > 0, 1, np.where(df['Close_pct'] < 0, -1, 0))

print(f'Số dòng : {len(df):,}')
print(f'Từ ngày : {df["Date"].min().date()}')
print(f'Đến ngày: {df["Date"].max().date()}')
print('✅ Dữ liệu sẵn sàng!')

---
## Q1 — Sau khi MBB đạt ĐỈNH 52 tuần, giá tiếp tục tăng hay ĐẢO CHIỀU?

### ❓ Làm như thế nào?
1. Tính `max_52w` = giá cao nhất trong **252 phiên giao dịch** trước (~1 năm)
2. Ngày đạt đỉnh 52 tuần = `Close > max_52w` lần đầu tiên vượt
3. Với mỗi ngày đó, tính `% thay đổi giá` sau **5, 10, 20 phiên**
4. Vẽ biểu đồ phân phối + tỷ lệ tăng/giảm

### 💡 Tại sao làm vậy?
> **52 tuần (~1 năm)** là mốc tâm lý quan trọng nhất trong đầu tư — toàn bộ trader đều nhìn vào nó.
> Có 2 trường phái: **Momentum** (vượt đỉnh → tiếp tục tăng) và **Mean reversion** (vượt đỉnh → điều chỉnh).
> Dữ liệu MBB sẽ cho biết trường phái nào đúng với cổ phiếu này.
> Dùng 252 phiên vì đó là số phiên giao dịch trung bình của 1 năm (loại ngày nghỉ).

In [ ]:
WINDOW_52W = 252
df['Max_52w'] = df['Close'].shift(1).rolling(WINDOW_52W).max()
df['Is_52w_high'] = (df['Close'] > df['Max_52w']) & df['Max_52w'].notna()

peak_idx = []
prev_peak = False
for i, row in df.iterrows():
    if row['Is_52w_high'] and not prev_peak:
        peak_idx.append(i)
    prev_peak = row['Is_52w_high']

HORIZONS = [5, 10, 20]
forward_returns = {h: [] for h in HORIZONS}
peak_details = []

for idx in peak_idx:
    row_data = {'Ngày đỉnh': df.loc[idx,'Date'].date(), 'Giá đỉnh': round(df.loc[idx,'Close'],2)}
    for h in HORIZONS:
        fi = idx + h
        if fi < len(df):
            ret = (df.loc[fi,'Close'] - df.loc[idx,'Close']) / df.loc[idx,'Close'] * 100
            forward_returns[h].append(ret)
            row_data[f'+{h} phiên (%)'] = round(ret, 2)
    peak_details.append(row_data)

peak_df = pd.DataFrame(peak_details)
print(f'=== Tìm thấy {len(peak_idx)} lần đạt đỉnh 52 tuần ===')
print(peak_df.to_string(index=False))
print()
for h in HORIZONS:
    rets = forward_returns[h]
    n_tang = sum(1 for r in rets if r > 0)
    print(f'+{h} phiên | TB: {np.mean(rets):+.2f}% | Tăng: {n_tang}/{len(rets)} ({n_tang/len(rets)*100:.0f}%)')

fig = plt.figure(figsize=(16, 10))
ax_main = fig.add_subplot(2, 3, (1, 3))
ax_main.plot(df['Date'], df['Close'], color='#2196F3', linewidth=1, alpha=0.7, label='Giá Close')
ax_main.scatter(df.loc[peak_idx,'Date'], df.loc[peak_idx,'Close'],
                color='#FF5722', s=80, zorder=5, label='Đỉnh 52 tuần', marker='^')
ax_main.set_title(f'Các lần MBB đạt Đỉnh 52 Tuần ({len(peak_idx)} lần)')
ax_main.set_ylabel('Giá (VNĐ)'); ax_main.legend()
ax_main.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax_main.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax_main.xaxis.get_majorticklabels(), rotation=30, fontsize=8)

colors_h = ['#42A5F5','#66BB6A','#FFA726']
for i, (h, color) in enumerate(zip(HORIZONS, colors_h)):
    ax = fig.add_subplot(2, 3, 4 + i)
    rets = forward_returns[h]
    n_tang = sum(1 for r in rets if r > 0)
    bar_colors = ['#4CAF50' if r > 0 else '#F44336' for r in rets]
    ax.bar(range(len(rets)), sorted(rets), color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.axhline(np.mean(rets), color=color, linewidth=2, label=f'TB: {np.mean(rets):+.2f}%')
    ax.set_title(f'+{h} phiên sau đỉnh\nTăng: {n_tang}/{len(rets)} ({n_tang/len(rets)*100:.0f}%)')
    ax.set_ylabel('% thay đổi'); ax.legend(fontsize=9)

plt.suptitle('Q1 — Sau khi MBB đạt đỉnh 52 tuần: Tiếp tục tăng hay đảo chiều?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Q1_dinh_52_tuan.png', bbox_inches='tight')
plt.show()
print('✅ Đã lưu: Q1_dinh_52_tuan.png')

---
## Q2 — Tháng nào MBB có nhiều ngày TĂNG nhất? (Tỷ lệ ngày xanh)

### ❓ Làm như thế nào?
1. Gán **nhãn xanh = 1** nếu `Close > Open` trong phiên, **đỏ = 0** nếu ngược lại
2. Nhóm theo **tháng (T1→T12)**, tính `% ngày xanh = sum(nhãn) / count`
3. Vẽ **heatmap tháng × năm** để thấy pattern nhất quán
4. Vẽ biểu đồ cột tổng hợp 5 năm

### 💡 Tại sao làm vậy?
> So sánh `Close vs Open` (trong phiên) chính xác hơn `Close vs Close hôm trước`
> vì phản ánh **lực mua/bán trong chính phiên đó**, không bị ảnh hưởng bởi gap giá.
> Heatmap giúp thấy tháng nào **nhất quán qua nhiều năm** — đó mới là pattern thực sự.
> Nhãn 1/0 đơn giản hóa dữ liệu phức tạp thành câu hỏi có/không dễ tổng hợp.

In [ ]:
df['Xanh'] = (df['Close'] > df['Open']).astype(int)

monthly_green = df.groupby('Month').agg(
    Tong_phien=('Xanh','count'), Ngay_xanh=('Xanh','sum')
).reset_index()
monthly_green['Pct_xanh'] = monthly_green['Ngay_xanh'] / monthly_green['Tong_phien'] * 100

thang_tot  = monthly_green.loc[monthly_green['Pct_xanh'].idxmax()]
thang_xau  = monthly_green.loc[monthly_green['Pct_xanh'].idxmin()]
thang_viet = {1:'T1',2:'T2',3:'T3',4:'T4',5:'T5',6:'T6',
              7:'T7',8:'T8',9:'T9',10:'T10',11:'T11',12:'T12'}

print('=== Tỷ lệ ngày TĂNG (Close > Open) theo tháng ===')
for _, r in monthly_green.iterrows():
    bar = '█' * int(r['Pct_xanh'] // 5)
    print(f"  Tháng {int(r['Month']):>2}: {r['Pct_xanh']:5.1f}% xanh | {bar}")
print(f"\n📈 Tháng xanh nhất: Tháng {int(thang_tot['Month'])} ({thang_tot['Pct_xanh']:.1f}%)")
print(f"📉 Tháng đỏ nhất  : Tháng {int(thang_xau['Month'])} ({thang_xau['Pct_xanh']:.1f}%)")

heat_green = df.groupby(['Year','Month'])['Xanh'].mean() * 100
heat_green = heat_green.unstack(level='Month')
heat_green.columns = [f'T{c}' for c in heat_green.columns]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

colors_bar = ['#4CAF50' if v >= 50 else '#F44336' for v in monthly_green['Pct_xanh']]
bars = ax1.bar([f"T{m}" for m in monthly_green['Month']], monthly_green['Pct_xanh'],
               color=colors_bar, edgecolor='black', linewidth=0.7)
ax1.axhline(50, color='gray', linestyle='--', linewidth=1.5, label='Ngưỡng 50%')
for bar, val in zip(bars, monthly_green['Pct_xanh']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Q2 — Tỷ lệ ngày tăng theo tháng (xanh ≥50% = tháng thuận)')
ax1.set_ylabel('% Ngày xanh'); ax1.set_ylim(0, 80); ax1.legend()

sns.heatmap(heat_green, annot=True, fmt='.1f', cmap='RdYlGn',
            center=50, vmin=20, vmax=80, linewidths=0.5, ax=ax2,
            cbar_kws={'label':'% Ngày xanh'})
ax2.set_title('Heatmap % ngày xanh theo Tháng × Năm')
ax2.set_xlabel('Tháng'); ax2.set_ylabel('Năm')

plt.tight_layout()
plt.savefig('Q2_ty_le_ngay_xanh.png', bbox_inches='tight')
plt.show()
print('✅ Đã lưu: Q2_ty_le_ngay_xanh.png')

---
## Q3 — Bóng nến trên (High–Close) CÓ DỰ BÁO ngày hôm sau giảm không?

### ❓ Làm như thế nào?
1. Tính `Tỷ lệ bóng trên = (High - Close) / (High - Low)` từ 0 đến 1
2. Phân loại: **bóng lớn** = tỷ lệ > 0.6 | **bóng nhỏ** = tỷ lệ < 0.2
3. Gán nhãn hôm sau: **1** = tăng, **0** = giảm
4. So sánh tỷ lệ tăng hôm sau giữa nhóm bóng lớn vs bóng nhỏ

### 💡 Tại sao làm vậy?
> **Bóng nến trên** = khoảng giá thử lên cao nhưng bị **từ chối** — người bán chiếm ưu thế.
> Bóng càng dài → áp lực bán ở vùng giá cao càng mạnh → khả năng giảm hôm sau cao hơn.
> Dùng ngưỡng 0.6 vì đó là khi bóng chiếm hơn 60% biên độ ngày — tín hiệu rõ ràng.
> Kỹ thuật phân tích nến Nhật (Candlestick) được kiểm chứng bằng dữ liệu thực.

In [ ]:
NGUONG_LON = 0.6
NGUONG_NHO = 0.2

df['Next_up'] = (df['Close_pct'].shift(-1) > 0).astype(int)

bong_lon = df[df['Shadow_ratio'] >= NGUONG_LON]
bong_nho = df[df['Shadow_ratio'] <= NGUONG_NHO]
tat_ca   = df[df['Shadow_ratio'].notna()]

def tinh_ty_le(grp, label):
    n = len(grp); n_up = grp['Next_up'].sum()
    pct = n_up / n * 100 if n > 0 else 0
    print(f'  [{label}] n={n:>4} | Hôm sau TĂNG: {pct:.1f}% | GIẢM: {100-pct:.1f}%')
    return pct

print('=== Bóng nến trên và tỷ lệ hôm sau TĂNG/GIẢM ===')
pct_lon = tinh_ty_le(bong_lon, f'Bóng LỚN (>={NGUONG_LON:.0%})')
pct_nho = tinh_ty_le(bong_nho, f'Bóng NHỎ (<={NGUONG_NHO:.0%})')
pct_all = tinh_ty_le(tat_ca,   'Tất cả phiên        ')
print(f'\n💡 Chênh lệch: bóng nhỏ tăng nhiều hơn {pct_nho - pct_lon:+.1f}% so với bóng lớn')

bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
lbls = ['0–20%','20–40%','40–60%','60–80%','80–100%']
df['Shadow_bin'] = pd.cut(df['Shadow_ratio'], bins=bins, labels=lbls)
bin_stats = df.groupby('Shadow_bin', observed=True)['Next_up'].agg(['count','mean']).reset_index()
bin_stats['mean'] *= 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bar_c = ['#4CAF50' if v >= 50 else '#F44336' for v in bin_stats['mean']]
bars = axes[0].bar(bin_stats['Shadow_bin'], bin_stats['mean'], color=bar_c, edgecolor='black')
axes[0].axhline(50, color='black', linestyle='--', linewidth=1.5)
for bar, row in zip(bars, bin_stats.itertuples()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{row.mean:.1f}%\n(n={row.count})', ha='center', fontsize=8)
axes[0].set_title('Tỷ lệ hôm sau TĂNG\ntheo dải bóng nến trên')
axes[0].set_xlabel('Tỷ lệ bóng trên'); axes[0].set_ylabel('% hôm sau tăng'); axes[0].set_ylim(0, 80)

nhom = [f'Bóng lớn\n(≥60%)\nn={len(bong_lon)}',
        f'Tất cả\nn={len(tat_ca)}',
        f'Bóng nhỏ\n(≤20%)\nn={len(bong_nho)}']
vals = [pct_lon, pct_all, pct_nho]
cols = ['#F44336','#90A4AE','#4CAF50']
b2 = axes[1].bar(nhom, vals, color=cols, edgecolor='black')
axes[1].axhline(50, color='black', linestyle='--', linewidth=1.5)
for bar, val in zip(b2, vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('So sánh 3 nhóm bóng nến'); axes[1].set_ylabel('% hôm sau tăng'); axes[1].set_ylim(0, 80)

axes[2].hist(df['Shadow_ratio'].dropna(), bins=40, color='#42A5F5', edgecolor='white', alpha=0.8)
axes[2].axvline(NGUONG_LON, color='red', linestyle='--', linewidth=2, label=f'Bóng lớn (≥{NGUONG_LON:.0%})')
axes[2].axvline(NGUONG_NHO, color='green', linestyle='--', linewidth=2, label=f'Bóng nhỏ (≤{NGUONG_NHO:.0%})')
axes[2].set_title('Phân phối tỷ lệ bóng nến trên')
axes[2].set_xlabel('(High–Close)/(High–Low)'); axes[2].set_ylabel('Số phiên'); axes[2].legend(fontsize=8)

plt.suptitle('Q3 — Bóng nến trên lớn có báo hiệu ngày hôm sau giảm không?', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Q3_bong_nen_tren.png', bbox_inches='tight')
plt.show()
print('✅ Đã lưu: Q3_bong_nen_tren.png')

---
## Q4 — Phiên ĐẦU THÁNG và phiên CUỐI THÁNG, phiên nào thường TĂNG hơn?

### ❓ Làm như thế nào?
1. Dùng `groupby('YearMonth').first()` lấy **phiên đầu tiên** mỗi tháng
2. Dùng `groupby('YearMonth').last()` lấy **phiên cuối cùng** mỗi tháng
3. Tính `% thay đổi Close_pct` của phiên đó (so với hôm trước)
4. So sánh tỷ lệ tăng + giá trị trung bình giữa 2 nhóm

### 💡 Tại sao làm vậy?
> **Cuối tháng**: quỹ đầu tư thực hiện 'window dressing' — mua vào cổ phiếu tốt để làm đẹp báo cáo.
> **Đầu tháng**: dòng tiền lương/tiết kiệm mới đổ vào thị trường.
> Đây là 'Turn of the Month Effect' — hiệu ứng được nghiên cứu trong học thuật tài chính.
> Kiểm chứng trực tiếp trên MBB xem cổ phiếu này có tuân theo hay không.

In [ ]:
dau_thang  = df.groupby('YearMonth').first().reset_index()
cuoi_thang = df.groupby('YearMonth').last().reset_index()

dau_thang['Label']  = 'Đầu tháng'
cuoi_thang['Label'] = 'Cuối tháng'
combined = pd.concat([dau_thang, cuoi_thang])
combined['Tang'] = (combined['Close_pct'] > 0).astype(int)

stats = combined.groupby('Label').agg(
    So_phien=('Close_pct','count'),
    TB_pct  =('Close_pct','mean'),
    Ty_le_tang=('Tang','mean')
).reset_index()
stats['Ty_le_tang'] *= 100

print('=== So sánh Phiên Đầu Tháng vs Cuối Tháng ===')
print(stats.to_string(index=False))
winner = stats.loc[stats['TB_pct'].idxmax(), 'Label']
print(f'\n🏆 Phiên thường tăng hơn: {winner}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

data_box = [dau_thang['Close_pct'].dropna().tolist(), cuoi_thang['Close_pct'].dropna().tolist()]
bp = axes[0].boxplot(data_box, labels=['Đầu tháng','Cuối tháng'], patch_artist=True)
bp['boxes'][0].set_facecolor('#42A5F5'); bp['boxes'][1].set_facecolor('#FFA726')
for med in bp['medians']: med.set(color='red', linewidth=2)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Phân phối % thay đổi\nĐầu tháng vs Cuối tháng'); axes[0].set_ylabel('% Thay đổi')

bar_c2 = ['#42A5F5','#FFA726']
bars3 = axes[1].bar(stats['Label'], stats['Ty_le_tang'], color=bar_c2, edgecolor='black')
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1.5)
for bar, val in zip(bars3, stats['Ty_le_tang']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[1].set_title('Tỷ lệ ngày TĂNG\nĐầu tháng vs Cuối tháng'); axes[1].set_ylabel('% Ngày tăng'); axes[1].set_ylim(0,80)

axes[2].plot(dau_thang['Date'], dau_thang['Close_pct'].rolling(3).mean(),
             color='#42A5F5', linewidth=2, label='Đầu tháng (MA3)')
axes[2].plot(cuoi_thang['Date'], cuoi_thang['Close_pct'].rolling(3).mean(),
             color='#FFA726', linewidth=2, label='Cuối tháng (MA3)')
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[2].set_title('Xu hướng theo thời gian (MA3)'); axes[2].set_ylabel('% thay đổi'); axes[2].legend(fontsize=8)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[2].xaxis.set_major_locator(mdates.YearLocator())

plt.suptitle('Q4 — Phiên đầu tháng hay cuối tháng MBB thường tăng hơn?', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Q4_dau_cuoi_thang.png', bbox_inches='tight')
plt.show()
print('✅ Đã lưu: Q4_dau_cuoi_thang.png')

---
## Q5 — Sau 3 phiên TĂNG liên tiếp, MBB có xu hướng ĐẢO CHIỀU không?
## (Kiểm chứng Mean Reversion)

### ❓ Làm như thế nào?
1. Gán nhãn từng phiên: **1** = tăng, **-1** = giảm, **0** = đứng giá
2. Duyệt toàn bộ dữ liệu, tìm chuỗi **3 số 1 liên tiếp** (3 phiên tăng)
3. Xem **phiên thứ 4**: gán **1** = tiếp tục tăng, **0** = đảo chiều giảm
4. Làm tương tự với 3 phiên GIẢM (-1)
5. Tổng hợp → kết luận MBB có theo Mean Reversion không

### 💡 Tại sao làm vậy?
> Áp dụng đúng kỹ thuật gán nhãn 1/0 của thầy nhưng ở cấp độ nâng cao hơn:
> **không chỉ tìm chuỗi mà còn dự báo bước tiếp theo** dựa trên chuỗi đó.
> Mean Reversion là lý thuyết: giá đi quá xa sẽ quay về trung bình.
> Chọn 3 phiên vì đủ để xác nhận xu hướng nhưng không quá dài để mẫu vẫn nhiều.

In [ ]:
DO_DAI = 3

def phan_tich_dao_chieu(signal_list, chieu, do_dai):
    ket_qua = []
    for i in range(do_dai, len(signal_list) - 1):
        if all(signal_list[i - do_dai + k] == chieu for k in range(do_dai)):
            phien_sau = signal_list[i]
            ket_qua.append({
                'Vi_tri'   : i,
                'Dao_chieu': int(phien_sau != chieu and phien_sau != 0),
                'Tiep_tuc' : int(phien_sau == chieu)
            })
    return pd.DataFrame(ket_qua)

sig      = df['Signal'].tolist()
kq_tang  = phan_tich_dao_chieu(sig,  1, DO_DAI)
kq_giam  = phan_tich_dao_chieu(sig, -1, DO_DAI)

def in_ket_qua(kq, ten):
    n = len(kq)
    if n == 0:
        print(f'{ten}: không tìm thấy chuỗi'); return 0, 0
    n_dao = kq['Dao_chieu'].sum(); n_tiep = kq['Tiep_tuc'].sum()
    p_dao = n_dao/n*100; p_tiep = n_tiep/n*100
    print(f'\n{ten}')
    print(f'  Tổng lần xuất hiện : {n}')
    print(f'  Phiên sau ĐẢO CHIỀU: {n_dao:>3} ({p_dao:.1f}%)')
    print(f'  Phiên sau TIẾP TỤC : {n_tiep:>3} ({p_tiep:.1f}%)')
    print(f'  → {"⚠️ CÓ xu hướng đảo chiều" if p_dao > 55 else "➡️ Không rõ xu hướng đảo chiều"}')
    return p_dao, p_tiep

print('=== KIỂM CHỨNG MEAN REVERSION — MBB ===')
pct_dao_t, pct_tt_t = in_ket_qua(kq_tang, f'Sau {DO_DAI} phiên TĂNG liên tiếp:')
pct_dao_g, pct_tt_g = in_ket_qua(kq_giam, f'Sau {DO_DAI} phiên GIẢM liên tiếp:')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def ve_pie_timeline(ax_pie, ax_time, kq, label_c, df_ref):
    if len(kq) == 0:
        ax_pie.text(0.5,0.5,'Không đủ dữ liệu',ha='center',va='center',transform=ax_pie.transAxes); return
    n_d = kq['Dao_chieu'].sum(); n_t = kq['Tiep_tuc'].sum(); n_k = len(kq)-n_d-n_t
    sizes = [s for s in [n_d,n_t,n_k] if s>0]
    lbls  = [l for l,s in zip([f'Đảo chiều\n{n_d} ({n_d/len(kq)*100:.0f}%)',
                                f'Tiếp tục\n{n_t} ({n_t/len(kq)*100:.0f}%)',
                                f'Đứng\n{n_k}'],
                               [n_d,n_t,n_k]) if s>0]
    clrs = [c for c,s in zip(['#F44336','#4CAF50','#90A4AE'],[n_d,n_t,n_k]) if s>0]
    ax_pie.pie(sizes, labels=lbls, colors=clrs, startangle=90)
    ax_pie.set_title(f'Sau {DO_DAI} phiên {label_c} liên tiếp\nPhiên thứ {DO_DAI+1} ra sao?')

    ax_time.plot(df_ref['Date'], df_ref['Close'], color='#90A4AE', linewidth=1, alpha=0.6)
    dao_idx  = kq[kq['Dao_chieu']==1]['Vi_tri'].tolist()
    tiep_idx = kq[kq['Tiep_tuc']==1]['Vi_tri'].tolist()
    valid_dao  = [i for i in dao_idx  if i < len(df_ref)]
    valid_tiep = [i for i in tiep_idx if i < len(df_ref)]
    if valid_dao:
        ax_time.scatter(df_ref.loc[valid_dao,'Date'], df_ref.loc[valid_dao,'Close'],
                        color='#F44336', s=50, zorder=5, label='Đảo chiều', marker='v')
    if valid_tiep:
        ax_time.scatter(df_ref.loc[valid_tiep,'Date'], df_ref.loc[valid_tiep,'Close'],
                        color='#4CAF50', s=50, zorder=5, label='Tiếp tục', marker='^')
    ax_time.set_title(f'Vị trí sau 3 phiên {label_c}')
    ax_time.set_ylabel('Giá Close'); ax_time.legend(fontsize=8)
    ax_time.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax_time.xaxis.set_major_locator(mdates.YearLocator())

ve_pie_timeline(axes[0,0], axes[0,1], kq_tang, 'TĂNG', df)
ve_pie_timeline(axes[1,0], axes[1,1], kq_giam, 'GIẢM', df)

plt.suptitle('Q5 — Mean Reversion: Sau 3 phiên tăng/giảm liên tiếp, MBB đảo chiều không?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Q5_mean_reversion.png', bbox_inches='tight')
plt.show()
print('✅ Đã lưu: Q5_mean_reversion.png')

---
## 📋 Tổng kết 5 câu hỏi

In [ ]:
n_peak_tang = sum(1 for r in forward_returns[10] if r > 0)
n_peak_all  = len(forward_returns[10])

print('=' * 65)
print('   TỔNG KẾT — CỔ PHIẾU MBB 2021–2026')
print('=' * 65)
print(f"""
Q1 | ĐỈNH 52 TUẦN
   Sau 10 phiên: {n_peak_tang}/{n_peak_all} lần tiếp tục tăng
   → {'Momentum mạnh — nên mua theo đỉnh' if n_peak_tang/n_peak_all > 0.5 else 'Dễ đảo chiều — thận trọng mua đỉnh'}

Q2 | TỶ LỆ NGÀY XANH
   Tháng xanh nhất: Tháng {int(thang_tot['Month'])} ({thang_tot['Pct_xanh']:.1f}%)
   Tháng đỏ nhất  : Tháng {int(thang_xau['Month'])} ({thang_xau['Pct_xanh']:.1f}%)

Q3 | BÓNG NẾN TRÊN
   Bóng lớn → hôm sau tăng: {pct_lon:.1f}%
   Bóng nhỏ → hôm sau tăng: {pct_nho:.1f}%
   → {'Bóng lớn CÓ cảnh báo giảm hôm sau' if pct_lon < pct_nho - 3 else 'Tín hiệu không rõ với MBB'}

Q4 | ĐẦU vs CUỐI THÁNG
   Phiên tốt hơn: {winner}

Q5 | MEAN REVERSION
   Sau 3 phiên tăng → đảo chiều: {pct_dao_t:.1f}%
   Sau 3 phiên giảm → đảo chiều: {pct_dao_g:.1f}%
   → {'Mean Reversion TỒN TẠI!' if pct_dao_t > 55 or pct_dao_g > 55 else 'MBB không theo Mean Reversion rõ ràng'}
""")
print('=' * 65)
print('✅ Hoàn thành toàn bộ phân tích!')